# KPI 14 — Supplier-Product Performance

**requirements.md (Supplier-Product Performance, KPI 14):** "Supplier-product performance: best combinations."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_14_supplier_product_performance` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cells below once (or after any change to the fact/dimension definitions).

**Definitions (state formulas):**

* **Grain:** one row per (supplier_id, product_id) sold on valid dates — 313 rows, the same 313 bridge pairs as KPI 4 (all bridge pairs sold; verified 2026-11-20). Pair ranking is this view's job; margin per pair was KPI 4's.
* **Identity:** `supplier_name` / `country` via `dim_suppliers.company_id` → `dim_companies` **current SCD2 version** (`__END_AT IS NULL`, KPI 13 convention); `sku` / `product_name` / `category` via `dim_products` current version (KPI 4 convention). 0 null attributes.
* **Measures:** revenue bases = KPI 1 (gross = all statuses + `order_date_valid`; net = excluding `cancelled` + `order_date_valid`). COGS / profit / margin = KPI 4 formulas exactly: COGS net-basis per line = `COALESCE(line_total * supplier_price / unit_price, quantity * supplier_price)`, `supplier_price` AS-OF `order_created_at` from the SCD2 bridge with first-version fallback; `gross_profit` = ROUND(net − COGS, 2); `gross_margin_pct` = ROUND(100 · gross_profit / net, 2) from the rounded profit so self-consistency is exact. `units_sold` = net-basis `SUM(quantity)` (valid-date null-qty lines stay in revenue but not units — KPI 3/4 convention); `line_count` / `order_count` valid-date gross.
* **Performance:** `net_revenue_rank` = ROW_NUMBER by net revenue DESC (tie-break supplier_id, product_id; 0 ties today); `net_revenue_share_pct` = ROUND(100·net / total net, 2) — the 313 shares sum to **100.05** (2-dp row-wise rounding artifact; KPI 13's 25 shares summed to 99.99); `cumulative_net_revenue_share_pct` = running share in rank order (100.00 at rank 313); `is_top_20_pct_pair` = rank ≤ CEIL(0.2·313) = 63 (KPI 2/3 top-20% Pareto convention — those 63 pairs hold 76.49% of net); `is_top_10_pair` = rank ≤ 10 (42.31% of net); `gross_profit_rank` = ROW_NUMBER by gross profit DESC; **`is_best_combination`** = top-20% by net revenue **AND** `gross_margin_pct` ≥ portfolio margin 26.85% (= 100·total profit / total net) — 30 pairs holding 38.64% of net.

**Verified 2026-11-20:** 313 rows = 313 distinct pairs = KPI 4 row count; net 28,333,498.48 / gross 30,723,269.53 EXACT KPI 1 tie; units 292,166 = KPI 4 tie; all 8 shared pair measures match the KPI 4 view exactly; both rank columns contiguous 1..313; 313/313 FULL match on all 22 columns via independent recompute.

In [0]:
%sql
-- KPI 14 view: supplier-product performance, best combinations. Idempotent.
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_14_supplier_product_performance
TBLPROPERTIES (
  'comment' = 'KPI 14 - Supplier-product performance, best combinations (requirements.md: Supplier-Product Performance, KPI 14). Grain: one row per (supplier_id, product_id) sold on valid dates - 313 rows, the same 313 bridge pairs as KPI 4 (all bridge pairs sold). Identity: supplier_name/country via dim_suppliers.company_id -> dim_companies CURRENT SCD2 version (__END_AT IS NULL, KPI 13 convention); sku/product_name/category via dim_products current version (KPI 4 convention); 0 null attributes. Measures: revenue bases = KPI 1 (gross = all statuses + order_date_valid; net = excluding cancelled + order_date_valid); COGS/gross_profit/gross_margin_pct = KPI 4 formulas (COGS net-basis per line = COALESCE(line_total * supplier_price / unit_price, quantity * supplier_price), supplier_price AS-OF order_created_at from the SCD2 bridge with first-version fallback; margin computed from the rounded profit so self-consistency is exact); units_sold = net-basis SUM(quantity) (valid-date null-qty lines stay in revenue but not units, KPI 3/4 convention); line_count/order_count valid-date gross. Performance: net_revenue_rank = ROW_NUMBER net DESC (tie-break supplier_id, product_id; 0 ties today); net_revenue_share_pct = ROUND(100*net/total net,2) - the 313 shares sum to 100.05 (2-dp row-wise rounding artifact); cumulative_net_revenue_share_pct = running share in rank order (100.00 at rank 313); is_top_20_pct_pair = rank <= CEIL(0.2*313)=63 (KPI 2/3 top-20% Pareto convention; those 63 pairs hold 76.49% of net); is_top_10_pair = rank <= 10 (42.31% of net); gross_profit_rank = ROW_NUMBER profit DESC; is_best_combination = top-20% by net AND gross_margin_pct >= portfolio margin 26.85% (100*total profit/total net) - 30 pairs holding 38.64% of net. Verified 2026-11-20: 313 rows = 313 distinct pairs = KPI 4 row count; net 28,333,498.48 / gross 30,723,269.53 EXACT KPI 1 tie; units 292,166 = KPI 4 tie; all 8 shared pair measures match the KPI 4 view exactly; ranks contiguous 1..313 on both rank columns; 313/313 FULL match on all 22 columns via independent recompute.'
)
AS
WITH lines AS (
  SELECT
    f.order_id,
    f.order_created_at,
    f.status,
    f.quantity,
    f.unit_price,
    f.line_total,
    d.product_id,
    s.supplier_id
  FROM b2b_ecommerce.gold.fact_order_items f
  JOIN b2b_ecommerce.gold.dim_products  d ON f.product_sk  = d.product_sk
  JOIN b2b_ecommerce.gold.dim_suppliers  s ON f.supplier_sk = s.supplier_sk
  WHERE f.order_date_valid
),
priced AS (
  SELECT
    l.*,
    COALESCE(a.supplier_price, fv.supplier_price) AS supplier_price
  FROM lines l
  LEFT JOIN b2b_ecommerce.gold.bridge_supplier_products a
    ON  a.supplier_id = l.supplier_id
    AND a.product_id = l.product_id
    AND a.__START_AT <= l.order_created_at
    AND (a.__END_AT IS NULL OR l.order_created_at < a.__END_AT)
  LEFT JOIN (
    SELECT b.supplier_id, b.product_id, b.supplier_price
    FROM b2b_ecommerce.gold.bridge_supplier_products b
    JOIN (
      SELECT supplier_id, product_id, MIN(__START_AT) AS ms
      FROM b2b_ecommerce.gold.bridge_supplier_products
      GROUP BY supplier_id, product_id
    ) m
      ON  b.supplier_id = m.supplier_id
      AND b.product_id  = m.product_id
      AND b.__START_AT  = m.ms
  ) fv
    ON  fv.supplier_id = l.supplier_id
    AND fv.product_id  = l.product_id
),
per_pair AS (
  SELECT
    p.supplier_id,
    p.product_id,
    COUNT(*)                                                              AS line_count,
    COUNT(DISTINCT p.order_id)                                            AS order_count,
    SUM(p.line_total)                                                     AS gross_revenue,
    SUM(CASE WHEN p.status <> 'cancelled' THEN p.line_total ELSE 0 END)   AS net_revenue,
    SUM(CASE WHEN p.status <> 'cancelled'
             THEN COALESCE(p.line_total * p.supplier_price / p.unit_price,
                          p.quantity * p.supplier_price)
             ELSE 0 END)                                                  AS cogs_val,
    SUM(CASE WHEN p.status <> 'cancelled' THEN p.quantity END)            AS units_sold
  FROM priced p
  GROUP BY p.supplier_id, p.product_id
),
attributed AS (
  SELECT
    pp.supplier_id,
    c.name       AS supplier_name,
    c.country,
    pp.product_id,
    dp.sku,
    dp.name      AS product_name,
    dp.category,
    pp.line_count,
    pp.order_count,
    pp.units_sold,
    pp.gross_revenue,
    pp.net_revenue,
    ROUND(pp.cogs_val, 2)                                       AS cogs,
    ROUND(pp.net_revenue - pp.cogs_val, 2)                     AS gross_profit,
    ROUND(100 * ROUND(pp.net_revenue - pp.cogs_val, 2)
          / pp.net_revenue, 2)                                 AS gross_margin_pct
  FROM per_pair pp
  LEFT JOIN b2b_ecommerce.gold.dim_suppliers s  ON s.supplier_id  = pp.supplier_id
  LEFT JOIN b2b_ecommerce.gold.dim_companies  c ON c.company_id  = s.company_id AND c.__END_AT IS NULL
  LEFT JOIN b2b_ecommerce.gold.dim_products  dp ON dp.product_id  = pp.product_id AND dp.__END_AT IS NULL
),
ranked AS (
  SELECT
    a.*,
    ROW_NUMBER() OVER (ORDER BY a.net_revenue DESC, a.supplier_id ASC, a.product_id ASC) AS net_revenue_rank,
    ROW_NUMBER() OVER (ORDER BY a.gross_profit DESC, a.supplier_id ASC, a.product_id ASC) AS gross_profit_rank,
    COUNT(*)    OVER ()                                                                   AS pair_count,
    SUM(a.net_revenue) OVER ()                                                            AS total_net_revenue,
    SUM(a.net_revenue) OVER (ORDER BY a.net_revenue DESC, a.supplier_id ASC, a.product_id ASC
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)           AS cumulative_net_revenue,
    ROUND(100 * SUM(a.gross_profit) OVER () / SUM(a.net_revenue) OVER (), 2)              AS portfolio_margin_pct
  FROM attributed a
)
SELECT
  supplier_id, supplier_name, country,
  product_id, sku, product_name, category,
  line_count, order_count, units_sold,
  gross_revenue, net_revenue,
  cogs, gross_profit, gross_margin_pct,
  net_revenue_rank,
  ROUND(100 * net_revenue / total_net_revenue, 2)            AS net_revenue_share_pct,
  ROUND(100 * cumulative_net_revenue / total_net_revenue, 2) AS cumulative_net_revenue_share_pct,
  net_revenue_rank <= CEIL(0.2 * pair_count)                 AS is_top_20_pct_pair,
  net_revenue_rank <= 10                                     AS is_top_10_pair,
  gross_profit_rank,
  (net_revenue_rank <= CEIL(0.2 * pair_count)
   AND gross_margin_pct >= portfolio_margin_pct)              AS is_best_combination
FROM ranked

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 313 rows = 313 distinct
-- (supplier, product) pairs = KPI 4 row count (all bridge pairs sold on
-- valid dates); 25 distinct suppliers / 160 distinct products; SUM(net_revenue)
-- 28,333,498.48 and SUM(gross_revenue) 30,723,269.53 = EXACT KPI 1 tie;
-- SUM(units_sold) 292,166 = KPI 4 units tie; SUM(cogs) 20,725,008.21 and
-- SUM(gross_profit) 7,608,490.27 = KPI 4 totals; portfolio margin 26.85%;
-- shares sum to 100.05 (2-dp row-wise rounding artifact); cumulative share
-- 100.00 at rank 313; both rank columns contiguous 1..313; 63 top-20% /
-- 10 top-10 / 30 best-combination flags, top-20% pairs holding 76.49% of
-- net and best combinations 38.64%; 0 null keys, names or attributes.
SELECT
  COUNT(*) AS rows_pairs,
  COUNT(DISTINCT CONCAT(supplier_id, '|', product_id)) AS distinct_pairs,
  COUNT(DISTINCT supplier_id) AS distinct_suppliers,
  COUNT(DISTINCT product_id) AS distinct_products,
  (SELECT COUNT(*) FROM b2b_ecommerce.gold.kpi_4_gross_margin) AS kpi4_row_count,
  ROUND(SUM(net_revenue), 2) AS total_net,
  ROUND(SUM(gross_revenue), 2) AS total_gross,
  (SELECT ROUND(SUM(net_revenue), 2) FROM b2b_ecommerce.gold.kpi_1_monthly_revenue_trend) AS kpi1_net_check,
  (SELECT ROUND(SUM(gross_revenue), 2) FROM b2b_ecommerce.gold.kpi_1_monthly_revenue_trend) AS kpi1_gross_check,
  SUM(units_sold) AS total_units,
  (SELECT SUM(units_sold) FROM b2b_ecommerce.gold.kpi_4_gross_margin) AS kpi4_units_check,
  ROUND(SUM(cogs), 2) AS total_cogs,
  ROUND(SUM(gross_profit), 2) AS total_profit,
  (SELECT ROUND(SUM(cogs), 2) FROM b2b_ecommerce.gold.kpi_4_gross_margin) AS kpi4_cogs_check,
  (SELECT ROUND(SUM(gross_profit), 2) FROM b2b_ecommerce.gold.kpi_4_gross_margin) AS kpi4_profit_check,
  ROUND(100 * SUM(gross_profit) / SUM(net_revenue), 2) AS portfolio_margin_pct,
  ROUND(SUM(net_revenue_share_pct), 2) AS sum_shares_pct,
  MAX(CASE WHEN net_revenue_rank = 313 THEN cumulative_net_revenue_share_pct END) AS cum_share_at_rank313,
  MIN(net_revenue_rank) AS min_rank, MAX(net_revenue_rank) AS max_rank,
  COUNT(DISTINCT net_revenue_rank) AS distinct_ranks,
  MIN(gross_profit_rank) AS min_profit_rank, MAX(gross_profit_rank) AS max_profit_rank,
  COUNT(DISTINCT gross_profit_rank) AS distinct_profit_ranks,
  SUM(CAST(is_top_20_pct_pair AS INT)) AS top20_pairs,
  SUM(CAST(is_top_10_pair AS INT)) AS top10_pairs,
  SUM(CAST(is_best_combination AS INT)) AS best_combinations,
  ROUND(100 * SUM(CASE WHEN is_top_20_pct_pair THEN net_revenue ELSE 0 END)
        / SUM(net_revenue), 2) AS top20_net_share_pct,
  ROUND(100 * SUM(CASE WHEN is_best_combination THEN net_revenue ELSE 0 END)
        / SUM(net_revenue), 2) AS best_comb_net_share_pct,
  SUM(CASE WHEN supplier_id IS NULL OR product_id IS NULL OR supplier_name IS NULL
            OR country IS NULL OR sku IS NULL OR product_name IS NULL
            OR category IS NULL THEN 1 ELSE 0 END) AS null_attribute_rows
FROM b2b_ecommerce.gold.kpi_14_supplier_product_performance

In [0]:
%sql
-- Verification: independently recompute every pair row through a different
-- path - measures aggregated at (supplier_sk, product_sk) grain FIRST and
-- only then mapped to business keys via the dims (vs the view's
-- join-then-aggregate at business-key grain), both ranks via correlated
-- COUNT subqueries and cumulative net via a self-join on the sort key
-- (net DESC, supplier_id ASC, product_id ASC) - all vs the view's window
-- functions - then FULL OUTER JOIN against the stored view and compare all
-- 22 per-pair columns (7 identity + 15 measures/flags).
-- Expected: 313 rows compared, 0 only on either side, 0 mismatches.
WITH lines AS (
  SELECT
    f.order_id, f.order_created_at, f.status, f.quantity, f.unit_price, f.line_total,
    d.product_id, s.supplier_id, f.product_sk, f.supplier_sk
  FROM b2b_ecommerce.gold.fact_order_items f
  JOIN b2b_ecommerce.gold.dim_products  d ON f.product_sk  = d.product_sk
  JOIN b2b_ecommerce.gold.dim_suppliers  s ON f.supplier_sk = s.supplier_sk
  WHERE f.order_date_valid
),
priced AS (
  SELECT
    l.*,
    COALESCE(a.supplier_price, fv.supplier_price) AS supplier_price
  FROM lines l
  LEFT JOIN b2b_ecommerce.gold.bridge_supplier_products a
    ON  a.supplier_id = l.supplier_id
    AND a.product_id = l.product_id
    AND a.__START_AT <= l.order_created_at
    AND (a.__END_AT IS NULL OR l.order_created_at < a.__END_AT)
  LEFT JOIN (
    SELECT b.supplier_id, b.product_id, b.supplier_price
    FROM b2b_ecommerce.gold.bridge_supplier_products b
    JOIN (
      SELECT supplier_id, product_id, MIN(__START_AT) AS ms
      FROM b2b_ecommerce.gold.bridge_supplier_products
      GROUP BY supplier_id, product_id
    ) m
      ON  b.supplier_id = m.supplier_id
      AND b.product_id  = m.product_id
      AND b.__START_AT  = m.ms
  ) fv
    ON  fv.supplier_id = l.supplier_id
    AND fv.product_id  = l.product_id
),
per_pair_sk AS (
  SELECT
    p.supplier_sk, p.product_sk,
    COUNT(*)                                                              AS line_count,
    COUNT(DISTINCT p.order_id)                                            AS order_count,
    SUM(p.line_total)                                                     AS gross_revenue,
    SUM(CASE WHEN p.status <> 'cancelled' THEN p.line_total ELSE 0 END)   AS net_revenue,
    SUM(CASE WHEN p.status <> 'cancelled'
             THEN COALESCE(p.line_total * p.supplier_price / p.unit_price,
                          p.quantity * p.supplier_price)
             ELSE 0 END)                                                  AS cogs_val,
    SUM(CASE WHEN p.status <> 'cancelled' THEN p.quantity END)            AS units_sold
  FROM priced p
  GROUP BY p.supplier_sk, p.product_sk
),
mapped AS (
  SELECT
    s.supplier_id, c.name AS supplier_name, c.country,
    dp.product_id, dp.sku, dp.name AS product_name, dp.category,
    pp.line_count, pp.order_count, pp.units_sold,
    pp.gross_revenue, pp.net_revenue,
    ROUND(pp.cogs_val, 2) AS cogs,
    ROUND(pp.net_revenue - pp.cogs_val, 2) AS gross_profit,
    ROUND(100 * ROUND(pp.net_revenue - pp.cogs_val, 2) / pp.net_revenue, 2) AS gross_margin_pct
  FROM per_pair_sk pp
  JOIN b2b_ecommerce.gold.dim_suppliers s  ON s.supplier_sk = pp.supplier_sk
  LEFT JOIN b2b_ecommerce.gold.dim_companies c ON c.company_id = s.company_id AND c.__END_AT IS NULL
  JOIN b2b_ecommerce.gold.dim_products dp ON dp.product_sk = pp.product_sk AND dp.__END_AT IS NULL
),
recomputed AS (
  SELECT
    m.*,
    (SELECT COUNT(*) + 1 FROM mapped m2
      WHERE m2.net_revenue > m.net_revenue
         OR (m2.net_revenue = m.net_revenue
             AND (m2.supplier_id < m.supplier_id
                  OR (m2.supplier_id = m.supplier_id AND m2.product_id < m.product_id)))) AS net_revenue_rank,
    (SELECT COUNT(*) + 1 FROM mapped m2
      WHERE m2.gross_profit > m.gross_profit
         OR (m2.gross_profit = m.gross_profit
             AND (m2.supplier_id < m.supplier_id
                  OR (m2.supplier_id = m.supplier_id AND m2.product_id < m.product_id)))) AS gross_profit_rank,
    (SELECT SUM(m2.net_revenue) FROM mapped m2
      WHERE m2.net_revenue > m.net_revenue
         OR (m2.net_revenue = m.net_revenue
             AND (m2.supplier_id < m.supplier_id
                  OR (m2.supplier_id = m.supplier_id AND m2.product_id <= m.product_id)))) AS cumulative_net_revenue,
    (SELECT SUM(m2.net_revenue) FROM mapped m2) AS total_net_revenue,
    (SELECT ROUND(100 * SUM(m2.gross_profit) / SUM(m2.net_revenue), 2) FROM mapped m2) AS portfolio_margin_pct
  FROM mapped m
),
final AS (
  SELECT
    supplier_id, supplier_name, country, product_id, sku, product_name, category,
    line_count, order_count, units_sold, gross_revenue, net_revenue,
    cogs, gross_profit, gross_margin_pct,
    net_revenue_rank, gross_profit_rank,
    ROUND(100 * net_revenue / total_net_revenue, 2) AS net_revenue_share_pct,
    ROUND(100 * cumulative_net_revenue / total_net_revenue, 2) AS cumulative_net_revenue_share_pct,
    net_revenue_rank <= 63 AS is_top_20_pct_pair,
    net_revenue_rank <= 10 AS is_top_10_pair,
    (net_revenue_rank <= 63 AND gross_margin_pct >= portfolio_margin_pct) AS is_best_combination
  FROM recomputed
)
SELECT
  COUNT(*) AS rows_compared,
  SUM(CASE WHEN r.supplier_id IS NULL THEN 1 ELSE 0 END) AS only_in_view,
  SUM(CASE WHEN f.supplier_id IS NULL THEN 1 ELSE 0 END) AS only_in_recompute,
  SUM(CASE WHEN r.supplier_id IS NOT NULL AND f.supplier_id IS NOT NULL AND NOT (
         r.supplier_name <=> f.supplier_name AND r.country <=> f.country
         AND r.sku <=> f.sku AND r.product_name <=> f.product_name
         AND r.category <=> f.category
         AND r.line_count <=> f.line_count AND r.order_count <=> f.order_count
         AND r.units_sold <=> f.units_sold
         AND r.gross_revenue <=> f.gross_revenue AND r.net_revenue <=> f.net_revenue
         AND r.cogs <=> f.cogs AND r.gross_profit <=> f.gross_profit
         AND r.gross_margin_pct <=> f.gross_margin_pct
         AND r.net_revenue_rank <=> f.net_revenue_rank AND r.gross_profit_rank <=> f.gross_profit_rank
         AND r.net_revenue_share_pct <=> f.net_revenue_share_pct
         AND r.cumulative_net_revenue_share_pct <=> f.cumulative_net_revenue_share_pct
         AND r.is_top_20_pct_pair <=> f.is_top_20_pct_pair
         AND r.is_top_10_pair <=> f.is_top_10_pair
         AND r.is_best_combination <=> f.is_best_combination) THEN 1 ELSE 0 END) AS measure_mismatches
FROM b2b_ecommerce.gold.kpi_14_supplier_product_performance f
FULL OUTER JOIN final r
  ON r.supplier_id = f.supplier_id AND r.product_id = f.product_id

In [0]:
%sql
-- Headline: top-10 supplier x product combinations by net revenue with
-- share, cumulative share, profit and margin, plus the portfolio constants
-- (window aggregates on every row): pair_count 313, total net 28,333,498.48,
-- portfolio margin 26.85%, top-20% (63 pairs) net share 76.49%,
-- best-combination count 30 holding 38.64% of net.
SELECT net_revenue_rank, supplier_name, country, sku, product_name, category,
       order_count, units_sold, net_revenue, gross_profit, gross_margin_pct,
       net_revenue_share_pct, cumulative_net_revenue_share_pct,
       is_best_combination,
       COUNT(*) OVER () AS pair_count,
       ROUND(SUM(net_revenue) OVER (), 2) AS total_net_revenue,
       ROUND(100 * SUM(gross_profit) OVER () / SUM(net_revenue) OVER (), 2) AS portfolio_margin_pct,
       ROUND(100 * SUM(CASE WHEN is_top_20_pct_pair THEN net_revenue ELSE 0 END) OVER ()
             / SUM(net_revenue) OVER (), 2) AS top20_net_share_pct,
       SUM(CAST(is_best_combination AS INT)) OVER () AS best_combination_count,
       ROUND(100 * SUM(CASE WHEN is_best_combination THEN net_revenue ELSE 0 END) OVER ()
             / SUM(net_revenue) OVER (), 2) AS best_comb_net_share_pct
FROM b2b_ecommerce.gold.kpi_14_supplier_product_performance
ORDER BY net_revenue_rank
LIMIT 10